# A_05 tiktoken and BPE

*Course appendix A: the toolbox (our own, not the book's appendix A). Dip in whenever you meet a tool you don't know yet.*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, write down what you think it will do. Then run it and compare.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll see `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `pyproject.toml`
>
> **This notebook:** In 02_03 you built a small BPE yourself. Real projects often use a ready-made one: does GPT-2's suit Homer?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- What tiktoken is
- How to use GPT-2's tokenizer on English and Greek
- Why GPT-2's tokenizer is a poor fit for Homer's Greek
- Why that's a reason to train our own BPE

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **tiktoken** | OpenAI's fast BPE tokenizer library | `import tiktoken` |
| **encoding** | in tiktoken: one finished vocabulary with its merges | `"gpt2"` |
| **byte** | the small basic unit computers store text in; one Greek letter takes 2 or 3 bytes | `α` = 2 bytes |
| **GPT-2** | an older language model by OpenAI; its tokenizer is the book's example | 50,257 tokens |

## 3. The idea

### ❓ What's tiktoken?

**tiktoken** is a library made by OpenAI (the company behind ChatGPT) that runs BPE tokenizers very fast. It comes with finished vocabularies, called encodings, including the one GPT-2 used. It's on our shopping list in `pyproject.toml`.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I load GPT-2's tokenizer?

`tiktoken.get_encoding("gpt2")`. The first time, it downloads a file from the internet and saves a copy (it's **cached**), so later runs work offline.

👉 **Predict first:** how many tokens does GPT-2's vocabulary have?

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print(enc.n_vocab)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`50257`. Compare our character vocabulary: 137.

</details>

### ❓ How many tokens does an English line take?

`enc.encode` turns text into token IDs, and `enc.decode` turns them back.

👉 **Predict first:** more or fewer tokens than characters?

In [ ]:
english = "Sing, O goddess, the anger of Achilles"
ids = enc.encode(english)
print(len(english), "characters ->", len(ids), "tokens")
print([enc.decode([i]) for i in ids])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

38 characters → **9 tokens**. Every word is a single token here, even ` Achilles` (with the space in front of it).

</details>

### ❓ And Homer's Greek?

👉 **Predict first:** the opening of the Iliad has 15 characters. More or fewer tokens?

In [ ]:
greek = "μῆνιν ἄειδε θεὰ"
ids = enc.encode(greek)
print(len(greek), "characters ->", len(ids), "tokens")
print(enc.decode(ids) == greek)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

15 characters → **20 tokens**. That's more tokens than characters! The round trip still works (`True`), but it's very wasteful.

</details>

### ❓ Why does Greek take more tokens than letters?

GPT-2's BPE learned its merges from mostly English web text. It almost never saw Homer's Greek, so it learned no Greek pieces to merge.

And it gets worse. When it has no merge, it falls back to **bytes**. A plain Greek letter is 2 bytes, and an accented one like `ῆ` is 3. So one Greek letter can become 3 tokens.

👉 **Predict first:** how many tokens is `ῆ`?

In [ ]:
for letter in ["α", "ῆ"]:
    ids = enc.encode(letter)
    print(letter, len(letter.encode("utf-8")), "bytes ->", len(ids), "tokens", ids)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`α` is 2 bytes but 1 token. `ῆ` is 3 bytes and **3 tokens**. `.encode("utf-8")` shows how a computer stores the letter as bytes.

</details>

## 5. Full example: GPT-2's tokenizer on Book 1

The whole of Odyssey Book 1, counted three ways.

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_books

text = load_books(BOOKS_DIR)[0]
print(f"characters (our tokenizer): {len(text):,}")
print(f"GPT-2 tokens:               {len(enc.encode(text)):,}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

GPT-2's tokenizer needs **more** tokens than our simple one-character-per-token tokenizer. So for Homer, a ready-made English tokenizer makes things worse. That's the reason to train our own BPE on Greek, as in [02_03](../02_text_to_numbers/02_03_bpe.ipynb).

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ So is BPE bad for Greek?

No, BPE is fine. GPT-2's *merges* are the problem, because they were learned from English. A BPE trained on Homer would learn Greek pieces like `πολλ` and common endings.

### ❓ Would GPT-2's tokenizer work for the English translation?

Yes, very well, as the 9-token example shows. [07_05](../07_advanced_iliad_and_combining/07_05_english_and_greek.ipynb) counts it on Butler's English.

## 6. Recap

- tiktoken runs GPT-2's BPE: English 38 characters → 9 tokens, but Greek 15 characters → 20 tokens.
- GPT-2 learned its merges from English, so Greek falls back to bytes, up to 3 tokens per letter.
- For Homer's Greek, we should train our own BPE.

**Next:** [A_06 The other tools](A_06_other_tools.ipynb).